In [15]:
import numpy as np

data = np.random.uniform(size=(500, 10))  # 500 entities, each contains 10 features
label = np.random.randint(low=0, high=2, size=(500, ))  # binary target

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris

clf = RandomForestClassifier(n_estimators=10, max_depth=3, random_state=42)
clf.fit(data, label)

RandomForestClassifier(max_depth=3, n_estimators=10, random_state=42)

In [35]:
def count_tree_values(tree):
    return (
        tree.feature.nbytes +
        tree.threshold.nbytes +
        tree.children_left.nbytes +
        tree.children_right.nbytes +
        tree.value.nbytes
    )

def count_random_forest_values(forest):
    return sum(count_tree_values(est.tree_) for est in forest.estimators_)


count_random_forest_values(clf)

6528

In [11]:
clf.estimators_[0].tree_.children_left.nbytes

72

In [42]:
import lightgbm as lgb

train_data = lgb.Dataset(data, label=label)

param = {'max_depth': 3, 'objective': 'binary'}

num_round = 10
bst = lgb.train(param, train_data, num_round)

[LightGBM] [Info] Number of positive: 246, number of negative: 254
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000258 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1670
[LightGBM] [Info] Number of data points in the train set: 500, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.492000 -> initscore=-0.032003
[LightGBM] [Info] Start training from score -0.032003
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

In [30]:
bst.save_model('model.txt')

In [ ]:
import re

def calculate_tree_params_memory(file_path, int_size=8, float_size=8):
    int_count = 0
    float_count = 0
    reading_trees = False

    # regex to match numeric literals
    num_pattern = re.compile(r"[-+]?\d*\.\d+(?:[eE][-+]?\d+)?|[-+]?\d+\b")

    with open(file_path, 'r') as f:
        for line in f:
            # detect start of trees section
            if line.startswith('Tree='):
                reading_trees = True
                continue
            # stop if end of trees
            if reading_trees and line.strip().startswith('end of trees'):
                break

            if not reading_trees:
                continue

            # skip per-tree constants
            stripped = line.strip()
            if stripped.startswith('shrinkage=') or stripped.startswith('is_linear='):
                continue

            # parse numeric values in other lines
            if '=' in line:
                _, vals = line.split('=', 1)
                for num_str in num_pattern.findall(vals):
                    # classify as float if contains a dot or exponent
                    if '.' in num_str or 'e' in num_str.lower():
                        float_count += 1
                    else:
                        int_count += 1

    total_memory = int_count * int_size + float_count * float_size
    return total_memory


calculate_tree_params_memory(file_path="model.txt")

5680

In [43]:
bst.dump_model()

{'name': 'tree',
 'version': 'v4',
 'num_class': 1,
 'num_tree_per_iteration': 1,
 'label_index': 0,
 'max_feature_idx': 9,
 'objective': 'binary sigmoid:1',
 'average_output': False,
 'feature_names': ['Column_0',
  'Column_1',
  'Column_2',
  'Column_3',
  'Column_4',
  'Column_5',
  'Column_6',
  'Column_7',
  'Column_8',
  'Column_9'],
 'monotone_constraints': [],
 'feature_infos': {'Column_0': {'min_value': 0.0012289606146690393,
   'max_value': 0.9992215867342672,
   'values': []},
  'Column_1': {'min_value': 0.0049895904102453805,
   'max_value': 0.9961453836932896,
   'values': []},
  'Column_2': {'min_value': 0.0007183176192733232,
   'max_value': 0.9973512641994302,
   'values': []},
  'Column_3': {'min_value': 0.0004672155530323074,
   'max_value': 0.9990732253821296,
   'values': []},
  'Column_4': {'min_value': 0.001133485712233706,
   'max_value': 0.9992323854510228,
   'values': []},
  'Column_5': {'min_value': 0.0066406570081878336,
   'max_value': 0.9998246457949097,
 

In [33]:
np.array([1, 2, 3]).nbytes / 3

8.0